# IMC Prosperity 4 — Data Exploration

Load round datasets and backtest outputs. Visualize order books, price series, PnL curves.

**Colab**: Click `Runtime > Run all` after setting `DATASET_DIR` below.

In [ ]:
# --- Config ---
DATASET_DIR = "../datasets/round1/"  # Change per round
RUN_DIR = "../runs/latest/"           # Or a specific run ID

# Colab: uncomment to clone repo first
# !git clone --recursive https://github.com/YOUR_ORG/slu-imc-prosperity-4.git
# %cd slu-imc-prosperity-4
# DATASET_DIR = "datasets/round1/"
# RUN_DIR = "runs/latest/"

In [ ]:
import csv
import json
import os
from collections import defaultdict
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
except ImportError:
    !pip install matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "monospace",
})

print("Ready.")

## 1. Load Activity Log

In [ ]:
def load_activity(run_dir):
    """Load activity.csv (semicolon or comma delimited)."""
    path = Path(run_dir) / "activity.csv"
    if not path.exists():
        print(f"Not found: {path}")
        return []
    with open(path) as f:
        content = f.read()
    delim = ";" if ";" in content.split("\n")[0] else ","
    return list(csv.DictReader(content.splitlines(), delimiter=delim))

activity = load_activity(RUN_DIR)
print(f"Loaded {len(activity)} rows")
if activity:
    print(f"Columns: {list(activity[0].keys())}")
    products = sorted(set(r.get('product', 'UNKNOWN') for r in activity))
    print(f"Products: {products}")

## 2. PnL Over Time

In [ ]:
if activity:
    by_product = defaultdict(list)
    total_by_ts = defaultdict(float)
    for row in activity:
        product = row.get("product", "UNKNOWN")
        ts = int(row.get("timestamp", 0))
        pnl = float(row.get("profitLoss", 0) or 0)
        by_product[product].append((ts, pnl))
        total_by_ts[ts] += pnl

    fig, ax = plt.subplots()
    total_sorted = sorted(total_by_ts.items())
    ax.plot([t[0] for t in total_sorted], [t[1] for t in total_sorted],
            linewidth=2.5, label="TOTAL", color="black")
    for product, data in sorted(by_product.items()):
        data.sort()
        ax.plot([d[0] for d in data], [d[1] for d in data],
                linewidth=1, linestyle="--", label=product, alpha=0.7)
    ax.axhline(y=0, color="gray", linewidth=0.5)
    ax.set_title("PnL Over Time")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("PnL (seashells)")
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()
    plt.show()

## 3. Price Series Per Product

In [ ]:
if activity:
    for product in products:
        rows = [r for r in activity if r.get("product") == product]
        timestamps = [int(r["timestamp"]) for r in rows]
        mids = [float(r.get("midPrice", 0) or 0) for r in rows]

        if not any(m > 0 for m in mids):
            continue

        fig, ax = plt.subplots()
        ax.plot(timestamps, mids, linewidth=1, label="Mid")
        ax.set_title(f"{product} — Mid Price")
        ax.set_xlabel("Timestamp")
        ax.set_ylabel("Price")
        ax.legend()
        plt.tight_layout()
        plt.show()

## 4. Load Trades

In [ ]:
def load_trades(run_dir):
    path = Path(run_dir) / "trades.csv"
    if not path.exists():
        print(f"Not found: {path}")
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

trades = load_trades(RUN_DIR)
print(f"Loaded {len(trades)} trades")
if trades:
    print(f"Columns: {list(trades[0].keys())}")

## 5. Trade Execution Scatter

In [ ]:
if trades and activity:
    trade_products = sorted(set(t.get("symbol", t.get("product", "")) for t in trades))
    for product in trade_products:
        # Mid price background
        rows = [r for r in activity if r.get("product") == product]
        mid_ts = [int(r["timestamp"]) for r in rows]
        mid_px = [float(r.get("midPrice", 0) or 0) for r in rows]

        pt = [t for t in trades if t.get("symbol", t.get("product")) == product]
        buys = [(int(t["timestamp"]), float(t["price"])) for t in pt
                if int(t.get("quantity", 0)) > 0 or t.get("side") == "BUY"
                or "SUBMISSION" in t.get("buyer", "")]
        sells = [(int(t["timestamp"]), float(t["price"])) for t in pt
                 if int(t.get("quantity", 0)) < 0 or t.get("side") == "SELL"
                 or "SUBMISSION" in t.get("seller", "")]

        fig, ax = plt.subplots()
        if mid_ts:
            ax.plot(mid_ts, mid_px, color="gray", linewidth=0.5, alpha=0.5)
        if buys:
            ax.scatter([b[0] for b in buys], [b[1] for b in buys],
                       c="green", marker="^", s=20, alpha=0.6, label=f"Buy ({len(buys)})")
        if sells:
            ax.scatter([s[0] for s in sells], [s[1] for s in sells],
                       c="red", marker="v", s=20, alpha=0.6, label=f"Sell ({len(sells)})")
        ax.set_title(f"{product} — Trades")
        ax.legend()
        plt.tight_layout()
        plt.show()

## 6. Metrics Summary

In [ ]:
metrics_path = Path(RUN_DIR) / "metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print(f"No metrics.json found at {metrics_path}")
    print("Run: python scripts/analyze.py runs/<run_id>")